Exploring the data

In [22]:
# Importing necessary libraries
import pandas as pd # reading in as a df
import statsmodels.formula.api as smf # to run the OLS

In [ ]:
# Read the data in
file = "dataverse_files/UCT_FINAL_CLEAN.dta"
data = pd.read_stata(file)

In [9]:
# Quick look
print(f"The shape of the df is {data.shape}")
print(data.describe())

The shape of the df is (2880, 981)
           surveyid    femaleres      maleres      village        treat  \
count  2.880000e+03  2880.000000  2880.000000  2880.000000  2880.000000   
mean   1.568195e+09     0.500000     0.500000   152.438194     0.349306   
min    1.065403e+08     0.000000     0.000000    15.000000     0.000000   
25%    1.172235e+09     0.000000     0.000000    86.000000     0.000000   
50%    1.482530e+09     0.500000     0.500000   129.000000     0.000000   
75%    2.169777e+09     1.000000     1.000000   222.000000     1.000000   
max    2.989871e+09     1.000000     1.000000   303.000000     1.000000   
std    7.270573e+08     0.500087     0.500087    83.579866     0.476833   

         spillover  purecontrol  control_village  treatXfemalerec  \
count  2880.000000  2880.000000      2880.000000      2880.000000   
mean      0.350694     0.300000         0.300000         0.198611   
min       0.000000     0.000000         0.000000         0.000000   
25%       0.0

In [26]:
print(data[['surveyid', 'femaleres', 'maleres', 'village', 'treat', 
            'baselinedate', 'endlinedate', 
            'asset_total_ppp0','asset_total_ppp_full0','asset_total_ppp_miss0','asset_total_ppp1']
            ].head(11))

       surveyid  femaleres  maleres  village  treat baselinedate endlinedate  \
0   106540326.0          0      1.0       74    0.0   2011-04-29  2012-09-04   
1   106540326.0          1      0.0       74    0.0   2011-04-29  2012-09-04   
2   106769148.0          0      1.0       55    1.0   2011-05-19  2012-10-15   
3   106769148.0          1      0.0       55    1.0   2011-05-19  2012-10-15   
4   106799419.0          0      1.0       71    0.0   2011-06-09  2012-10-03   
5   106799419.0          1      0.0       71    0.0   2011-06-09  2012-10-03   
6   106854772.0          0      1.0       46    0.0   2011-05-09  2012-09-07   
7   106854772.0          1      0.0       46    0.0   2011-05-09  2012-09-07   
8   106979214.0          0      1.0       15    1.0   2011-05-26  2012-09-03   
9   106979214.0          1      0.0       15    1.0   2011-05-26  2012-09-03   
10  107079236.0          0      1.0       15    0.0   2011-05-26         NaT   

    asset_total_ppp0  asset_total_ppp_f

In [12]:
for col in data.columns:
    print(col)

surveyid
femaleres
maleres
village
treat
spillover
purecontrol
control_village
treatXfemalerec
treatXmalerec
treatXfemalerecXmarried
treatXmalerecXmarried
treatXsinglerec
treatXlump
treatXmonthly
treatXlarge
treatXsmall
treatXmalerecXlump
treatXmalerecXsmall
treatXlumpXsmall
treatXmalerecXlumpXsmall
treatXmalerecXlarge
treatXlumpXlarge
treatXmalerecXlumpXlarge
treatXmalerecXmonthly
treatXmonthlyXsmall
treatXmalerecXmonthlyXsmall
treatXmonthlyXlarge
treatXmalerecXmonthlyXlarge
treatXfemalerecXlump
treatXfemalerecXsmall
treatXfemalerecXlumpXsmall
treatXfemalerecXlarge
treatXfemalerecXlumpXlarge
treatXfemalerecXmonthly
treatXfemalerecXmonthlyXsmall
treatXfemalerecXmonthlyXlarge
treatXmaleres
treatXfemaleres
treatXlargeXmaleres
treatXlargeXfemaleres
treatXsmallXmaleres
treatXsmallXfemaleres
treatXlumpXmaleres
treatXlumpXfemaleres
treatXmonthlyXmaleres
treatXmonthlyXfemaleres
treatXmalerecXmaleres
treatXmalerecXfemaleres
treatXfemalerecXmaleres
treatXfemalerecXfemaleres
treatXlumpXearly
tre

In [ ]:
# Try and reproduce the baseline OLS (only done cols 1 and 2 thus far, no interaction treatments)
# (Need to fix outcomes education index, psych index, and female empowerment index)
"""
Basline Balance: y_vhib = a_v + B_0 + B_1.T_vh + e_vhib

- y_vhib: outcome with suffix 0 for value at baseline
- a_v: 'village'
- T_vh: 'treat'
"""

# Keeping only households that were were within village, not village comparisons
data_within = data[data['purecontrol'] == 0].copy()

# Drop duplicate observations (one male and female for each)
data = data_within[data_within['femaleres'] == 1].copy()

# Empty list
results = []

# Define all our outcome variables at baseline
outcomes_base = [
    'asset_total_ppp0',
    'cons_nondurable_ppp0',
    'ent_total_rev_ppp0',
    'fs_hhfoodindexnew0',
    'med_hh_healthindex0',
    'ed_index0',
    'psy_index_z0',
    'ih_overall_index_z0'
]

# Attempting to recreate table 1
for outcome in outcomes_base:
    # Run ols for each outcome
    model = smf.ols(f'{outcome} ~ treat + C(village)', data=data).fit()

    # Getting the control means as per table 1 and column 1 
    control_mean = data[data["spillover"] == 1][outcome].mean()
    control_sd = data[data["spillover"] == 1][outcome].std()

    results.append({
        'Outcome':        outcome,
        'Control Mean':   f"{control_mean:.2f}",
        'Control SD':     f"({control_sd:.2f})",
        'Treat Coef':     f"{model.params['treat']:.2f}",
        'SE':             f"({model.bse['treat']:.2f})",
        'P-value':        f"{model.pvalues['treat']:.3f}"
    })

results_df = pd.DataFrame(results) # makes results list into a df
print(results_df.to_string(index=False)) # prints results_df without index




             Outcome Control Mean Control SD Treat Coef      SE P-value
    asset_total_ppp0       383.36   (374.34)      -1.15 (24.36)   0.962
cons_nondurable_ppp0       181.99   (127.23)      -6.16  (8.23)   0.454
  ent_total_rev_ppp0        84.92   (402.79)     -33.19 (18.87)   0.079
  fs_hhfoodindexnew0        -0.00     (1.00)       0.00  (0.06)   0.997
 med_hh_healthindex0         0.01     (1.02)       0.03  (0.06)   0.683
           ed_index0         0.00     (1.00)      -0.07  (0.06)   0.275
        psy_index_z0        -0.00     (1.00)       0.04  (0.07)   0.557
 ih_overall_index_z0         0.00     (1.00)      -0.05  (0.07)   0.484


In [ ]:
# Try and reproduce the ATE in table 2 (for cols 1 and 2) - within village households
# (Need to fix outcomes education index, psych index, and female empowerment index)
"""
Basline Balance: y_vhiE = a_v + B_0 + B_1.T_vh + delta_1.y_vhib + delta_2.M_vhiB + e_vhiE

- y_vhiE: outcome with suffix 1 at endline (variable of interest)

- a_v: 'village' (village fixed effects)
- B_0: intercept which represents base village (intercept)
- T_vh: 'treat' (treatment)
- y_vhib: outcome with suffix 0 for value at baseline (a baseline control)
- M_vhiB: used as a dummy if baseline value is missing (a baseline control)
"""

# Keeping only households that were were within village, not village comparisons
data_within = data[data['purecontrol'] == 0].copy()

# Drop duplicate observations (one male and female for each)
data = data_within[data_within['femaleres'] == 1].copy()

# Drop the 68 missing endline values
data = data[data['asset_total_ppp1'] != 0].copy()

# Empty list
results = []

# Define all our outcome variables at endline - this is our variable of interest
outcomes_end = [
    'asset_total_ppp1',
    'cons_nondurable_ppp1',
    'ent_total_rev_ppp1',
    'fs_hhfoodindexnew1',
    'med_hh_healthindex1',
    'ed_index1',
    'psy_index_z1',
    'ih_overall_index_z1'
]

# Define all our outcome variables at baseline
outcomes_base = [
    'asset_total_ppp0',
    'cons_nondurable_ppp0',
    'ent_total_rev_ppp0',
    'fs_hhfoodindexnew0',
    'med_hh_healthindex0',
    'ed_index0',
    'psy_index_z0',
    'ih_overall_index_z0'
]

# Dummy for outcomes that are missing at baseline
outcomes_missing = [
    'asset_total_ppp_miss0',
    'cons_nondurable_ppp_miss0',
    'ent_total_rev_ppp_miss0',
    'fs_hhfoodindexnew_miss0',
    'med_hh_healthindex_miss0',
    'ed_index_miss0',
    'psy_index_z_miss0',
    'ih_overall_index_z_miss0'
]

# Attempting to recreate table 1
for outcome_end, outcome_base, outcome_miss in zip(outcomes_end, outcomes_base, outcomes_missing):
    model = smf.ols(f'{outcome_end} ~ C(village) + treat + {outcome_base} + {outcome_miss}', data=data).fit()

    # Getting the control means as per table 1 and column 1 
    control_mean = data[data["spillover"] == 1][outcome_end].mean()
    control_sd = data[data["spillover"] == 1][outcome_end].std()

    results.append({
        'Outcome':        outcome_end,
        'Control Mean':   f"{control_mean:.2f}",
        'Control SD':     f"({control_sd:.2f})",
        'Treat Coef':     f"{model.params['treat']:.2f}",
        'SE':             f"({model.bse['treat']:.2f})",
        'P-value':        f"{model.pvalues['treat']:.3f}"
    })

    # print(model.summary())

results_df = pd.DataFrame(results) # makes results list into a df
print(results_df.to_string(index=False)) # prints results_df without index




             Outcome Control Mean Control SD Treat Coef      SE P-value
    asset_total_ppp1       494.80   (415.32)     301.51 (27.45)   0.000
cons_nondurable_ppp1       157.61    (82.18)      35.66  (5.82)   0.000
  ent_total_rev_ppp1        48.98    (90.52)      16.15  (5.82)   0.006
  fs_hhfoodindexnew1         0.00     (1.00)       0.26  (0.06)   0.000
 med_hh_healthindex1        -0.00     (1.00)      -0.03  (0.06)   0.579
           ed_index1         0.00     (1.00)       0.08  (0.06)   0.152
        psy_index_z1        -0.00     (1.00)       0.23  (0.07)   0.001
 ih_overall_index_z1         0.00     (1.00)      -0.01  (0.07)   0.866


68
